# 古事類苑 OCR（v5b モデル）— RunPod 一括実行ノートブック

日文研 IIIF の画像を自動でダウンロードし、v5b モデルで OCR を行い、**山田形式（識別タグ付きテキスト）**に変換します。

## 使い方
1. 下の **「設定」** のセルで、処理したい冊名（`VOLUMES`）を確認・変更する
2. メニュー **Run → Run All Cells** を押す（カーネルの再起動は不要です）
3. 最後のセルに「✅ 完了」と表示されたら、左のファイル一覧から `results.zip` をダウンロード
4. **RunPod の画面で Pod を Terminate（削除）する**（忘れると課金が続きます）

目安：1冊（約1,400頁）で **約1時間**、RTX 4090 で約 1 ドル。
途中で赤い「❌」が表示されたら、その下のメッセージを開発者に送ってください。


In [ ]:
# ▶ 設定（ここだけ変更してください）
VOLUMES = ['bung_2']      # 日文研 IIIF の冊名。複数可：['bung_2', 'bung_3']（冊名の一覧は README を参照）
AUTO_STOP_HOURS = 1 + 1.5 * len(VOLUMES)   # この時間が過ぎたら Pod を自動停止（課金の止め忘れ防止）。通常は変更不要
PARALLEL = 3              # GPU 1 枚あたりの同時 OCR 数（RTX 3090 / 4090 なら 3）。GPU が複数あれば全部使います

REPO = 'https://github.com/oushiei120/kojiruien-ocr'
MODEL_ID = '1MYugrFHvdWlKnG1tsT-ylyZwo8fBL-kJ'   # v5b モデル（Google Drive）
MODEL_URL = REPO + '/releases/download/v5b/v5b_weights.pth'   # 同じモデルの予備（GitHub Releases）
W = '/workspace'
K = f'{W}/kojiruien-ocr'

import os, sys, subprocess
os.makedirs(W, exist_ok=True)

def run_live(cmd):
    """コマンドを実行し、出力をそのまま表示する。失敗したら止まる。"""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError('❌ 失敗しました。上のメッセージを開発者に送ってください: ' + ' '.join(map(str, cmd)))

print('対象：', VOLUMES)


In [ ]:
# ▶ 自動停止の設定（AUTO_STOP_HOURS 時間後に Pod を停止。結果は /workspace に残ります）
import shutil
pid = os.environ.get('RUNPOD_POD_ID')
if not pid or not shutil.which('runpodctl'):
    print('⚠️ 自動停止を設定できない環境です。終わったら手動で Pod を停止してください')
else:
    subprocess.run(['pkill', '-f', 'kojiruien_autostop'], check=False)
    cmd = (f'exec -a kojiruien_autostop bash -c "sleep {int(AUTO_STOP_HOURS * 3600)}; '
           f'runpodctl stop pod {pid} || runpodctl pod stop {pid}"')
    subprocess.Popen(['bash', '-c', cmd], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f'⏰ {AUTO_STOP_HOURS} 時間後に Pod を自動停止します')


In [ ]:
# ▶ 環境構築（初回 5〜10 分）＋ プログラムと v5b モデルのダウンロード
import os

os.makedirs('/workspace', exist_ok=True)
BASE_DIR = '/workspace'
os.chdir(BASE_DIR)

def run(cmd):
    print(f'  → {cmd[:80]}...' if len(cmd)>80 else f'  → {cmd}')
    r = os.system(cmd)
    if r != 0: print(f'  ⚠️ リターンコード {r}')
    return r

# ---- 00. Python / torch のバージョン確認（torch 2.1 + cu121 以外なら入れ替え）----
import sys, subprocess
print(f'🐍 Python {sys.version.split()[0]}')
if sys.version_info[:2] > (3, 11):
    raise SystemExit('❌ Python が新しすぎます（3.10 か 3.11 が必要）。README のとおり runpod/pytorch:2.1.1-py3.10-cuda12.1.1-devel-ubuntu22.04 のテンプレートを使ってください')
tv = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__)'],
                    capture_output=True, text=True).stdout.strip()
print(f'🔥 現在の torch: {tv or "なし"}')
if not (tv.startswith('2.1.') and 'cu121' in tv):
    print('🔧 [00] torch 2.1.2 + CUDA 12.1 に入れ替え中（約 3 分）...')
    run('pip install --no-cache-dir -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121')
    tv = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.cuda.is_available())'],
                        capture_output=True, text=True).stdout.strip()
    print(f'  ✅ 入れ替え後: {tv}')

# ---- 0. 🔥 packaging バージョン固定（最重要！最初に戻す）----
print('🔧 [0/6] packaging バージョン固定中...')
run('pip install --force-reinstall -q packaging==22.0')
run('pip install --force-reinstall -q lightning-utilities==0.10.0')

# ---- 1. numpy および関連パッケージのバージョン固定 ----
print('🔧 [1/6] numpy およびコア依存関係の競合を解消中...')
run('pip uninstall -y numpy pandas scikit-learn pytorch-lightning mmcv mmdet mmpretrain > /dev/null 2>&1')
run('pip install --no-cache-dir -q numpy==1.23.5 pandas==1.5.3 scikit-learn==1.2.2 pytorch-lightning==1.6.5')

# ---- 2. NDLOCR 依存関係 ----
print('🔧 [2/6] NDLOCR 依存関係をインストール中...')
run('pip install --no-cache-dir -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html')
run('pip install --no-cache-dir -q mmdet==3.3.0 mmpretrain==1.2.0')
run('pip install -q hydra-core hydra-colorlog omegaconf lxml==4.9.3 scikit-image lmdb nltk==3.6.2 rich pyrootutils pyyaml click joblib opencv-python gdown')

# ---- 3. 🔥 scikit-learn ABI 修正（最重要！最後に実行） ----
print('🔧 [3/6] scikit-learn ABI 修正中...')
run('pip install --no-cache-dir --force-reinstall -q numpy==1.23.5 scikit-learn==1.2.2')

# ---- 4. KyTea（ルビ振り機能、任意） ----
print('🔧 [4/6] KyTea を確認中...')
if os.system('python3 -c "import Mykytea" 2>/dev/null') != 0:
    run('apt-get update -qq && apt-get install -y -qq build-essential git autoconf automake libtool > /dev/null 2>&1')
    run('cd /workspace && (test -d kytea || git clone --depth 1 https://github.com/neubig/kytea.git) && cd kytea && autoreconf -i && ./configure && make -j4 && make install && ldconfig && pip install -q kytea')
    print('  ✅ KyTea コンパイル完了')
else:
    print('  ✅ KyTea インストール済み')

# ---- 5. モデルパッケージ ----
print('🔧 [5/6] モデルパッケージ...')
full_package_dir = os.path.join(BASE_DIR, 'full_package')
if not os.path.exists(full_package_dir):
    import gdown
    zip_path = os.path.join(BASE_DIR, 'kojiruien_full_package.zip')
    if not os.path.exists(zip_path):
        gdown.download('https://drive.google.com/uc?id=1nw8pgzwHZvv24v_5geq2tib8l_52dFvt', zip_path, quiet=False)
    run(f'apt-get install -y -qq unzip > /dev/null 2>&1 && cd {BASE_DIR} && unzip -q -o kojiruien_full_package.zip')
    if os.path.exists(zip_path): os.remove(zip_path)
    print('  ✅ 解凍完了（zip 削除済み）')
else:
    print('  ✅ 既存')

print('✅ 環境構築 完了', flush=True)

# プログラム（このリポジトリ）
if not os.path.exists(K):
    run_live(['git', 'clone', '--depth', '1', REPO, K])
else:
    run_live(['git', '-C', K, 'pull', '--ff-only'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'], check=True)

# v5b モデル（Google Drive。取得できなければ GitHub Releases から）
MODEL = f'{W}/v5b_weights.pth'
ok = lambda: os.path.exists(MODEL) and os.path.getsize(MODEL) > 300_000_000
if not ok():
    import gdown
    try:
        gdown.download(f'https://drive.google.com/uc?id={MODEL_ID}', MODEL, quiet=False)
    except Exception as e:
        print('Google Drive から取得できませんでした：', e, flush=True)
if not ok():
    print('GitHub から取得中（約 320 MB）...', flush=True)
    subprocess.run(['curl', '-L', '--fail', '-sS', '-o', MODEL, MODEL_URL])
if not ok():
    raise RuntimeError('❌ モデルをダウンロードできませんでした。しばらく待ってからこのセルを再実行してください')
print(f'✅ モデル {os.path.getsize(MODEL) / 1e6:.0f} MB', flush=True)
assert os.path.exists(f'{W}/full_package/ndlocr_cli'), '❌ NDLOCR のパッケージがありません'


In [ ]:
# ▶ 画像のダウンロード（日文研 IIIF。全冊を同時に取得、約 20〜30 分。30 秒ごとに進捗を表示）
import glob, re, time
procs = {}
for vol in VOLUMES:
    os.makedirs(f'{W}/run/{vol}', exist_ok=True)
    log = f'{W}/run/{vol}/fetch.log'
    procs[vol] = subprocess.Popen(
        [sys.executable, '-u', f'{K}/tools/fetch_iiif_pages.py',
         '--manifest', f'https://iiif.nichibun.ac.jp/IIIF/manifest/KJZ/{vol}.json',
         '--start', '1', '--end', '9999', '--out', f'{W}/run/{vol}/img', '--workers', '6'],
        stdout=open(log, 'w'), stderr=subprocess.STDOUT)

def total(vol):
    m = re.search(r'(\d+) 頁をダウンロード', open(f'{W}/run/{vol}/fetch.log', encoding='utf-8', errors='ignore').read())
    return m.group(1) if m else '?'

while True:
    time.sleep(30)
    states = {v: p.poll() for v, p in procs.items()}
    print(time.strftime('%H:%M'), '  '.join(f'{v}: {len(glob.glob(f"{W}/run/{v}/img/*.jpg"))} / {total(v)} 枚' for v in VOLUMES), flush=True)
    bad = [v for v, s in states.items() if s not in (None, 0)]
    if bad:
        for p in procs.values():
            p.kill()
        for v in bad:
            print(open(f'{W}/run/{v}/fetch.log', encoding='utf-8', errors='ignore').read()[-2000:])
        raise RuntimeError(f'❌ 画像のダウンロードに失敗しました（{bad}）。上のメッセージを開発者に送ってください')
    if all(s == 0 for s in states.values()):
        break
print('✅ ダウンロード完了', flush=True)


In [ ]:
# ▶ OCR（v5b、1 冊 約 20〜30 分（GPU 1 枚の場合）。30 秒ごとに進捗を表示）
for vol in VOLUMES:
    print(f'===== {vol} =====', flush=True)
    run_live([sys.executable, '-u', f'{K}/tools/run_ocr_parallel.py',
              f'{W}/run/{vol}/img', f'{W}/run/{vol}/work', f'{W}/run/{vol}/{vol}_v5b.sorted.xml',
              '--ckpt', MODEL, '--per-gpu', str(PARALLEL)])


In [ ]:
# ▶ 山田形式（識別タグ付きテキスト）への変換
for vol in VOLUMES:
    out = f'{W}/run/{vol}/{vol}_v5b.txt'
    run_live([sys.executable, '-u', f'{K}/src/convert.py', f'{W}/run/{vol}/{vol}_v5b.sorted.xml',
              '-o', out, '--lexicon', f'{K}/data/lexicon.json'])
    lines = open(out, encoding='utf-8').read().splitlines()
    cnt = {t: sum(1 for l in lines if l.startswith(t + ' ')) for t in 'PABCGVNHWMX'}
    print(f'  {vol}: 頁 {cnt["P"]}｜引用書名 V {cnt["V"]}・W {cnt["W"]}｜編目 N {cnt["N"]}・M {cnt["M"]}｜項名 C {cnt["C"]}', flush=True)


In [ ]:
# ▶ 結果をまとめる → 左のファイル一覧で results.zip を右クリック → Download
import zipfile, glob
out = f'{W}/results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for vol in VOLUMES:
        for f in (f'{W}/run/{vol}/{vol}_v5b.txt', f'{W}/run/{vol}/{vol}_v5b.sorted.xml'):
            if os.path.exists(f):
                z.write(f, os.path.basename(f))
print(f'✅ 完了：{out}（{os.path.getsize(out) / 1e6:.1f} MB）')
print('  *_v5b.txt        … 山田形式の識別タグ付きテキスト')
print('  *_v5b.sorted.xml … OCR の結果（枠の座標つき）')
print('ダウンロードしたら、RunPod の画面で Pod を Terminate してください。')
